# 03 — Hybrid Search & Reciprocal Rank Fusion

*Level 2 — Advanced RAG*

## Objective
Fuse dense and sparse rankings with RRF and check, honestly, whether it actually beats dense retrieval alone on this corpus.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "hybrid-search"))
sys.path.insert(0, str(LEVEL_DIR / "query-transformations"))
sys.path.insert(0, str(LEVEL_DIR / "metadata-filtering"))
sys.path.insert(0, str(LEVEL_DIR / "context-compression"))


In [2]:
from common.dataset import prepare
from retrieval.dense import DenseRetriever
from retrieval.sparse import BM25Retriever
from retrieval.top_k_experiments import sweep_top_k
from bm25_vector import HybridRetriever
from rrf import reciprocal_rank_fusion

data = prepare()
corpus_texts = {d: data.corpus_text(d) for d in data.doc_ids()}
dense = DenseRetriever.from_corpus(corpus_texts)
sparse = BM25Retriever.from_corpus(corpus_texts)
hybrid = HybridRetriever(dense, sparse)
print("retrievers ready")


retrievers ready


## Recall@K: dense vs. sparse vs. hybrid


In [3]:
k_values = (1, 3, 5, 10, 20)
dense_recall = sweep_top_k(dense, data.queries, data.qrels, k_values)
sparse_recall = sweep_top_k(sparse, data.queries, data.qrels, k_values)
hybrid_recall = sweep_top_k(hybrid, data.queries, data.qrels, k_values)

print(f"{'k':>4} {'dense':>8} {'sparse':>8} {'hybrid':>8}")
for k in k_values:
    print(f"{k:>4} {dense_recall[k]:>8.3f} {sparse_recall[k]:>8.3f} {hybrid_recall[k]:>8.3f}")


   k    dense   sparse   hybrid
   1    0.733    0.683    0.740
   3    0.873    0.807    0.850
   5    0.900    0.847    0.887
  10    0.917    0.867    0.923
  20    0.943    0.890    0.957


## RRF on a single query — see the fusion happen


In [4]:
# Picked because neither dense-only nor sparse-only individually ranks
# the relevant doc in their Top-5, but the fused list does -- a genuine
# example of hybrid search rescuing a miss.
sample_qid = "128"
sample_query = data.queries[sample_qid]
relevant = set(data.qrels[sample_qid])

dense_ranking = dense.search(sample_query, top_k=10)
sparse_ranking = sparse.search(sample_query, top_k=10)
fused = reciprocal_rank_fusion([dense_ranking, sparse_ranking])

print(f"Query: {sample_query!r}  (relevant: {relevant})\n")
print("Fused (RRF) ranking:")
for doc_id, score in fused[:10]:
    d_rank = next((i for i, (d, _) in enumerate(dense_ranking, 1) if d == doc_id), None)
    s_rank = next((i for i, (d, _) in enumerate(sparse_ranking, 1) if d == doc_id), None)
    mark = " <-- relevant" if doc_id in relevant else ""
    print(f"  rrf={score:.4f}  dense_rank={d_rank}  sparse_rank={s_rank}  {doc_id}{mark}")


Query: 'Arterioles have a larger lumen diameter than venules.'  (relevant: {'8290953'})

Fused (RRF) ranking:
  rrf=0.0328  dense_rank=1  sparse_rank=1  1122279
  rrf=0.0161  dense_rank=2  sparse_rank=None  22867765
  rrf=0.0161  dense_rank=None  sparse_rank=2  8290953 <-- relevant
  rrf=0.0159  dense_rank=3  sparse_rank=None  34876410
  rrf=0.0159  dense_rank=None  sparse_rank=3  791050
  rrf=0.0156  dense_rank=4  sparse_rank=None  52175065
  rrf=0.0156  dense_rank=None  sparse_rank=4  2576811
  rrf=0.0154  dense_rank=5  sparse_rank=None  4427392
  rrf=0.0154  dense_rank=None  sparse_rank=5  16237005
  rrf=0.0152  dense_rank=6  sparse_rank=None  10697096


## What I observed

Hybrid essentially **tied dense-only** on this corpus (both ≈0.94 Recall@20), and even trailed it slightly at a couple of middling K values, despite beating it at K=1 and K=20. That is *not* a bug — it's what happens when you fuse a strong ranker (dense, here) with a meaningfully weaker one (sparse, here): RRF pulls the strong ranker's confident top picks down slightly to make room for the weak ranker's opinion, which doesn't always pay off.

**Lesson:** hybrid search is a hedge, not a guaranteed win — it reduces the *risk* of one retriever's blind spot (BM25 misses paraphrases; dense misses exact rare terms) at a small cost when one side is already dominant. Measure it on your own corpus before assuming it helps; on a corpus with more exact-match queries (IDs, codes, names), expect hybrid to pull ahead of dense-only by a much larger margin.

## Next

[04 — Reranking](./04_reranking.ipynb)
